# Simulation 1 Cued Unguided Retrieval Figure

> Generate the Simulation 1 figure package for unguided retrieval with weak test-phase film cues.

This notebook crosses reminder condition with task encoding strength under unguided retrieval.
It holds weak VIT-style film cues constant and separates topline phase-total recall from serial-position and context diagnostics.

In [ ]:
import csv
import os
import warnings
from pathlib import Path

import numpy as np
from IPython.display import Markdown
from jax import random
from jaxcmr.analyses.spc import fixed_pres_spc
from jaxcmr.helpers import find_project_root

from selective_interference_v2 import (
    Paradigm,
    apply_supplied_item_context_cue,
    batch_trial,
    configure_rates,
    load_fit_params,
    make_factory,
    make_is_emotional,
    make_is_target,
    prepare_sweep,
    simulate_periodic_cued_free_recall,
    split_scales_for_cache,
    standard_remap,
    sweep_rngs,
)

warnings.filterwarnings("ignore")


In [ ]:
# --- sweep configuration ---
PROJECT_ROOT = ""
FIT_PATH = "results/fits/Dupertuys2026_eCMR_source_only_phi_no_shared_support_best_of_1.json"
FIGURE_DIR = ""
DATA_DIR = ""
FIGURE_STR = ""
RNG_SEED = 0

RASTER_N_TRIALS = 12
RASTER_MAX_OUTPUTS = 16
RASTER_SAMPLE_SEED_OFFSET = 1000

TASK_MCF_VALUES = [1.0, 2.0]
TASK_MCF_LABELS = {
    1.0: "Weak task encoding",
    2.0: "Strong task encoding",
}

PRIMARY_SOURCE_ORIENTATION = 0.0

FILM_CUE_REINSTATEMENT = 0.90
CUE_INTERVAL = 4
FIRST_CUE_AFTER = 0

N_FILM = 16
N_BREAK = 16
N_INTERFERENCE = 16
N_FILLER = 16
EXPERIMENT_COUNT = 5000
MAX_RECALL = 48

FILM_EMOTIONAL = True
INTERFERENCE_EMOTIONAL = False
CACHE_AFTER = "filler"

EXTRA_FIXED_SCALES = {
    "source_learning_baseline": 0.05,
    "primacy_scale": 0.1,
    "primacy_decay": 1.5,
}

BASE_FIXED_SCALES = {
    "break_drift_scale": 1.0,
    "break_mcf_scale": 1.0,
    "filler_drift_scale": 1.0,
    "filler_mcf_scale": 1.0,
}

REMINDER_CONDITIONS = {
    "No reminder": {
        "reminder_start_drift_scale": 0.0,
        "reminder_drift_scale": 0.0,
    },
    "With reminder": {
        "reminder_start_drift_scale": 1.0,
        "reminder_drift_scale": 1.0,
    },
}

REMINDER_LABELS = {
    "No reminder": "Without pre-task reminder",
    "With reminder": "With pre-task film reminder",
}

RETRIEVAL_FIXED_SCALES = {
    "start_drift_scale": 0.0,
    "tau_scale": 1.0,
    "target_recall_drift_scale": 1.0,
    "rejected_recall_drift_scale": 1.0,
    "recall_drift_scale": 1.0,
}


In [ ]:
#| output: asis
if PROJECT_ROOT:
    project_root = Path(PROJECT_ROOT)
else:
    project_root = Path(find_project_root())

fit_path = project_root / FIT_PATH
figure_dir = project_root / FIGURE_DIR if FIGURE_DIR else None
data_dir = project_root / DATA_DIR if DATA_DIR else None
task_mcf_values = np.asarray(TASK_MCF_VALUES, dtype=float)
extra_fixed_scales = {**EXTRA_FIXED_SCALES}
base_fixed_scales = {**BASE_FIXED_SCALES, **extra_fixed_scales}

params, n_subjects = load_fit_params(fit_path)
paradigm = Paradigm(
    n_film=N_FILM,
    n_break=N_BREAK,
    n_interference=N_INTERFERENCE,
    n_filler=N_FILLER,
    experiment_count=EXPERIMENT_COUNT,
    max_recall=MAX_RECALL,
)
factory = make_factory(
    is_emotional=make_is_emotional(
        paradigm,
        film_emotional=FILM_EMOTIONAL,
        interference_emotional=INTERFERENCE_EMOTIONAL,
    ),
    is_target=make_is_target(paradigm),
)
rng = random.PRNGKey(RNG_SEED)

Markdown(
    f"Loaded {n_subjects} fit parameter set(s). Running {len(REMINDER_CONDITIONS)} reminder conditions "
    f"and {len(task_mcf_values)} interference-strength values with a film cue before first recall "
    f"and {EXPERIMENT_COUNT} simulated trials per cell."
)


In [ ]:
def position_phase_labels(paradigm):
    return (
        ["film"] * paradigm.n_film
        + ["break"] * paradigm.n_break
        + ["task"] * paradigm.n_interference
        + ["filler"] * paradigm.n_filler
    )


def write_rows(path, fieldnames, rows):
    os.makedirs(path.parent, exist_ok=True)
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def phase_recall_totals(spc):
    totals = {}
    for phase in sorted(set(position_phase)):
        positions = [idx for idx, value in enumerate(position_phase) if value == phase]
        totals[phase] = float(np.sum(spc[positions]))
    return totals


position_phase = position_phase_labels(paradigm)
n_presented = len(position_phase)


In [ ]:
def cued_recall_trial(
    model,
    rng,
    film_items,
    cue_scale,
    max_recall,
    cue_interval,
    first_cue_after,
):
    recall_rng, initial_cue_rng = random.split(rng)
    model = model.start_retrieving()
    if first_cue_after == 0:
        cue_item = random.choice(initial_cue_rng, film_items)
        model = apply_supplied_item_context_cue(model, cue_item, cue_scale)
        scheduled_first_cue_after = cue_interval
    else:
        scheduled_first_cue_after = first_cue_after
    _, recalls, cue_attempts, cue_items = simulate_periodic_cued_free_recall(
        model,
        max_recall,
        recall_rng,
        film_items,
        cue_scale,
        cue_interval=cue_interval,
        first_cue_after=scheduled_first_cue_after,
    )
    return recalls, cue_attempts, cue_items


batched_cued_recall = batch_trial(cued_recall_trial, n_args=7, n_static=3)

prepared_cache = {}

for reminder_condition, reminder_scales in REMINDER_CONDITIONS.items():
    for task_scale in task_mcf_values:
        fixed_scales = {
            **base_fixed_scales,
            **reminder_scales,
            "interference_mcf_scale": float(task_scale),
        }
        pre_cache_scales, post_cache_scales = split_scales_for_cache(
            fixed_scales,
            CACHE_AFTER,
        )
        prepared = prepare_sweep(
            params,
            paradigm,
            factory,
            cache_after=CACHE_AFTER,
            **pre_cache_scales,
        )
        prepared_cache[(reminder_condition, float(task_scale))] = prepared


In [ ]:
cue_rows = []
cue_distribution_rows = []
cue_recall_sequences = {}
cue_rng = random.PRNGKey(RNG_SEED + 1)

for reminder_condition, reminder_scales in REMINDER_CONDITIONS.items():
    for task_scale in task_mcf_values:
        fixed_scales = {
            **base_fixed_scales,
            **reminder_scales,
            "interference_mcf_scale": float(task_scale),
        }
        _, post_cache_scales = split_scales_for_cache(
            fixed_scales,
            CACHE_AFTER,
        )
        prepared = prepared_cache[(reminder_condition, float(task_scale))]
        ready_models = configure_rates(
            prepared.models,
            **post_cache_scales,
            **RETRIEVAL_FIXED_SCALES,
            film_source_start_drift_rate=float(PRIMARY_SOURCE_ORIENTATION),
        )
        rngs, cue_rng = sweep_rngs(
            cue_rng,
            prepared.n_subjects,
            prepared.experiment_count,
        )
        recalls, cue_attempts, cue_items = batched_cued_recall(
            ready_models,
            rngs,
            paradigm.film_items,
            FILM_CUE_REINSTATEMENT,
            paradigm.max_recall,
            CUE_INTERVAL,
            FIRST_CUE_AFTER,
        )
        raw_recalls = np.asarray(recalls).reshape(-1, recalls.shape[-1])
        remapped_recalls = np.asarray(
            standard_remap(
                raw_recalls,
                paradigm,
                show_break=True,
            )
        )
        cue_recall_sequences[(reminder_condition, float(task_scale))] = remapped_recalls
        spc = np.asarray(fixed_pres_spc(remapped_recalls, n_presented), dtype=float)
        totals = phase_recall_totals(spc)
        for phase, total in totals.items():
            cue_rows.append({
                "reminder_condition": reminder_condition,
                "task_condition": TASK_MCF_LABELS[float(task_scale)],
                "task_mcf_scale": float(task_scale),
                "film_cue_reinstatement": FILM_CUE_REINSTATEMENT,
                "first_cue_after": FIRST_CUE_AFTER,
                "phase": phase,
                "recall_probability_mass": total,
            })
        for position, phase in enumerate(position_phase, start=1):
            cue_distribution_rows.append({
                "reminder_condition": reminder_condition,
                "reminder_label": REMINDER_LABELS[reminder_condition],
                "task_condition": TASK_MCF_LABELS[float(task_scale)],
                "task_mcf_scale": float(task_scale),
                "film_cue_reinstatement": FILM_CUE_REINSTATEMENT,
                "first_cue_after": FIRST_CUE_AFTER,
                "position": position,
                "phase": phase,
                "recall_probability": float(spc[position - 1]),
            })


In [ ]:
def cue_total(reminder_condition, task_scale, phase):
    for row in cue_rows:
        if (
            row["reminder_condition"] == reminder_condition
            and np.isclose(row["task_mcf_scale"], task_scale)
            and row["phase"] == phase
        ):
            return row["recall_probability_mass"]
    return np.nan


In [ ]:
SEQUENCE_ROWS_SUFFIX = "sequence_rows"
PHASE_TOTALS_SUFFIX = "phase_totals_sequence_summary"
SEQUENCE_SELECTION_SUFFIX = "sequence_selection_summary"
RECALL_DISTRIBUTIONS_SUFFIX = "recall_distributions"
SEQUENCE_SELECTION_RULE = "closest_to_condition_mean_film_count"

RASTER_PHASE_CODES = {
    "none": 0,
    "film": 1,
    "task": 2,
    "delay_filler": 3,
}


def recall_item_class(item_id):
    item_id = int(item_id)
    if item_id == 0:
        return "none"
    if 1 <= item_id <= paradigm.n_film:
        return "film"
    task_start = paradigm.n_film + paradigm.n_break + 1
    task_end = paradigm.n_film + paradigm.n_break + paradigm.n_interference
    if task_start <= item_id <= task_end:
        return "task"
    return "delay_filler"


def recall_phase_codes(recall_matrix):
    codes = np.zeros_like(recall_matrix, dtype=int)
    film_mask = (recall_matrix >= 1) & (recall_matrix <= paradigm.n_film)
    task_start = paradigm.n_film + paradigm.n_break + 1
    task_end = paradigm.n_film + paradigm.n_break + paradigm.n_interference
    task_mask = (recall_matrix >= task_start) & (recall_matrix <= task_end)
    other_mask = (recall_matrix != 0) & ~film_mask & ~task_mask
    codes[film_mask] = RASTER_PHASE_CODES["film"]
    codes[task_mask] = RASTER_PHASE_CODES["task"]
    codes[other_mask] = RASTER_PHASE_CODES["delay_filler"]
    return codes


def recall_phase_counts(recall_matrix):
    film_mask = (recall_matrix >= 1) & (recall_matrix <= paradigm.n_film)
    task_start = paradigm.n_film + paradigm.n_break + 1
    task_end = paradigm.n_film + paradigm.n_break + paradigm.n_interference
    task_mask = (recall_matrix >= task_start) & (recall_matrix <= task_end)
    nonzero_mask = recall_matrix != 0
    return {
        "film": np.sum(film_mask, axis=1),
        "task": np.sum(task_mask, axis=1),
        "delay_filler": np.sum(nonzero_mask & ~film_mask & ~task_mask, axis=1),
    }


def select_mean_film_trials(recalls, rng):
    counts = recall_phase_counts(recalls)
    film_counts = counts["film"]
    mean_film_count = float(np.mean(film_counts))
    distance = np.abs(film_counts - mean_film_count)
    tie_breakers = rng.random(recalls.shape[0])
    ranked_indices = np.lexsort((tie_breakers, distance))
    sample_size = min(RASTER_N_TRIALS, recalls.shape[0])
    selected_indices = np.sort(ranked_indices[:sample_size])
    rank_lookup = {int(trial_index): rank for rank, trial_index in enumerate(ranked_indices, start=1)}
    return selected_indices, rank_lookup, counts, mean_film_count, distance


def build_sequence_sample_rows():
    rng = np.random.default_rng(RNG_SEED + RASTER_SAMPLE_SEED_OFFSET)
    rows = []
    selection_rows = []
    for reminder_condition in REMINDER_CONDITIONS:
        for task_scale in task_mcf_values:
            key = (reminder_condition, float(task_scale))
            recalls = cue_recall_sequences[key]
            selected_indices, rank_lookup, counts, mean_film_count, distance = select_mean_film_trials(recalls, rng)
            sampled = recalls[selected_indices, :RASTER_MAX_OUTPUTS]
            codes = recall_phase_codes(sampled)
            for display_trial, trial_index in enumerate(selected_indices, start=1):
                trial_index = int(trial_index)
                film_count = int(counts["film"][trial_index])
                task_count = int(counts["task"][trial_index])
                delay_filler_count = int(counts["delay_filler"][trial_index])
                distance_from_mean = float(distance[trial_index])
                selection_rows.append({
                    "reminder_condition": reminder_condition,
                    "task_condition": TASK_MCF_LABELS[float(task_scale)],
                    "task_mcf_scale": float(task_scale),
                    "selection_rule": SEQUENCE_SELECTION_RULE,
                    "mean_film_count": mean_film_count,
                    "sampled_trial_index": trial_index,
                    "display_trial": display_trial,
                    "selection_rank": int(rank_lookup[trial_index]),
                    "film_count": film_count,
                    "task_count": task_count,
                    "delay_filler_count": delay_filler_count,
                    "distance_from_mean": distance_from_mean,
                })
                for output_position, item_id in enumerate(sampled[display_trial - 1], start=1):
                    item_class = recall_item_class(item_id)
                    rows.append({
                        "reminder_condition": reminder_condition,
                        "task_condition": TASK_MCF_LABELS[float(task_scale)],
                        "task_mcf_scale": float(task_scale),
                        "selection_rule": SEQUENCE_SELECTION_RULE,
                        "sampled_trial_index": trial_index,
                        "display_trial": display_trial,
                        "output_position": output_position,
                        "item_id": int(item_id),
                        "item_class": item_class,
                        "phase_code": int(codes[display_trial - 1, output_position - 1]),
                    })
    return rows, selection_rows


sequence_rows, sequence_selection_rows = build_sequence_sample_rows()

if figure_dir is not None:
    write_rows(
        data_dir / f"{FIGURE_STR}_{RECALL_DISTRIBUTIONS_SUFFIX}.csv",
        [
            "reminder_condition",
            "reminder_label",
            "task_condition",
            "task_mcf_scale",
            "film_cue_reinstatement",
            "first_cue_after",
            "position",
            "phase",
            "recall_probability",
        ],
        cue_distribution_rows,
    )
    write_rows(
        data_dir / f"{FIGURE_STR}_{SEQUENCE_ROWS_SUFFIX}.csv",
        [
            "reminder_condition",
            "task_condition",
            "task_mcf_scale",
            "selection_rule",
            "sampled_trial_index",
            "display_trial",
            "output_position",
            "item_id",
            "item_class",
            "phase_code",
        ],
        sequence_rows,
    )
    write_rows(
        data_dir / f"{FIGURE_STR}_{SEQUENCE_SELECTION_SUFFIX}.csv",
        [
            "reminder_condition",
            "task_condition",
            "task_mcf_scale",
            "selection_rule",
            "mean_film_count",
            "sampled_trial_index",
            "display_trial",
            "selection_rank",
            "film_count",
            "task_count",
            "delay_filler_count",
            "distance_from_mean",
        ],
        sequence_selection_rows,
    )
    write_rows(
        data_dir / f"{FIGURE_STR}_{PHASE_TOTALS_SUFFIX}.csv",
        [
            "reminder_condition",
            "task_condition",
            "task_mcf_scale",
            "film_cue_reinstatement",
            "first_cue_after",
            "phase",
            "recall_probability_mass",
        ],
        cue_rows,
    )


In [ ]:
print("Film cue before first recall")
for reminder_condition in REMINDER_CONDITIONS:
    for task_scale in task_mcf_values:
        film_total = cue_total(reminder_condition, float(task_scale), "film")
        task_total = cue_total(reminder_condition, float(task_scale), "task")
        print(
            f"  {reminder_condition}, {TASK_MCF_LABELS[float(task_scale)]}: "
            f"film={film_total:.3f}, task={task_total:.3f}"
        )
